In [108]:
import numpy as np
import scipy as sc

In [109]:
# Initialisation

N = 4 # Longueur du MPS
d = 2 # Nombre de paramètres physiques
chi = 5 # Bond dimension

A = [] # liste contenant le MPS 

#Convention: on numérote initialement les tenseurs dans le sens anti trigo, en partant de la branche à gauche

#Ajout du premier tenseur (rang 2)
A.append(np.random.rand(d,chi))

#Ajout des tenseurs "centraux" (rang 3)
for _ in range(N-2):
    A.append(np.random.rand(chi,d,chi))

#Ajout du dernier tenseur (rang 2)
A.append(np.random.rand(chi,d))






In [110]:
#Pour les tests, on prend H nul

#Ajout du premier MPO (rang 3)
H.append(np.zeros((d,1,d)))

#Ajout des MPO "centraux" (rang 3)
for _ in range(N-2):
    H.append(np.zeros((1,d,1,d)))

#Ajout du dernier MPO (rang 2)
H.append(np.zeros((1,d,d)))



Nous allons d'abord écrire les fonctions qui permettent de décomposer un hamiltonien sous forme d'un MPO

In [111]:
def contraction(H): #Si H est un MPO (une liste de tenseurs) cette fonction contracte le MPO. Sert pour les vérifications
    n=len(H)
    if n>=2:
        t=H[0]
        for i in range(n-1):
            t=np.tensordot(t,H[i+1],([i+1],[0])) #On doit décaler les indices de contraction car à chaque étape on "gagne" 2 ordres
            t=np.transpose(t,())
        return t
    else:
        return H

Pour construire H sous forme de MPO, nous utilisons la décomposition de Jordan-Wigner qui nous donne le hamiltonien décomposé en matrices de Pauli. Les matrices de Pauli sont les

In [112]:
Sx=np.array([[0,1],[1,0]], dtype="complex_")
Sy=np.array([[0,-1j],[1j,0]], dtype="complex_")
Sz=np.array([[1,0],[0,-1]], dtype="complex_")
I2=np.array([[1,0],[0,1]], dtype="complex_")

In [113]:
Splus=0.5*(Sx+1j*Sy)
Sminus=0.5*(Sx-1j*Sy)

In [114]:
Hx=np.zeros((1,2,1,2),dtype="complex_")
Hx[0,:,0,:]=Sx
Hxg=np.zeros((2,1,2),dtype="complex_")
Hxg[:,0,:]=Sx
Hxd=np.zeros((1,2,2),dtype="complex_")
Hxd[0,:,:]=Sx

In [115]:
Hy=np.zeros((1,2,1,2),dtype="complex_")
Hy[0,:,0,:]=Sy
Hyg=np.zeros((2,1,2),dtype="complex_")
Hyg[:,0,:]=Sy
Hyd=np.zeros((1,2,2),dtype="complex_")
Hyd[0,:,:]=Sy

In [116]:
Hz=np.zeros((1,2,1,2), dtype="complex_")
Hz[0,:,0,:]=Sz
Hzg=np.zeros((2,1,2),dtype="complex_")
Hzg[:,0,:]=Sz
Hzd=np.zeros((1,2,2),dtype="complex_")
Hzd[0,:,:]=Sz

In [117]:
H2=np.zeros((1,2,1,2),dtype="complex_")
H2[0,:,0,:]=I2
H2g=np.zeros((2,1,2),dtype="complex_")
H2g[:,0,:]=I2
H2d=np.zeros((1,2,2),dtype="complex_")
H2d[0,:,:]=I2

In [118]:
#Création du (MPO hamiltonien)

# prend deux MPO A et B en entrée (de même longueur) et renvoie le MPO correspondant à la somme de A et B
# Rq: les MPO sont supposés du même format que précedemment
def MPO_sum(A,B):
    n = len(A)
    chi_A = A[0].shape[1]
    chi_B = B[0].shape[1]
    chi_S = chi_A + chi_B
    S = []

    #Ajout du premier tenseur
    S_i = np.zeros((d,chi_S,d), dtype="complex_")
    for sigma1 in range(d):
        for sigma2 in range(d):

            for k in range(chi_A):
                S_i[sigma1,k,sigma2] = A[0][sigma1,k,sigma2]

            for k in range(chi_B):
                S_i[sigma1,k+chi_A,sigma2] = B[0][sigma1,k,sigma2]
            
    S.append(S_i)

    #Ajout des tenseurs "centraux"
    for i in range(1,n-1):
        S_i = np.zeros((chi_S,d,chi_S,d), dtype="complex_")
        for sigma1 in range(d):
            for sigma2 in range(d):
                
                for k in range(chi_A):
                    for l in range(chi_A):
                        S_i[k,sigma1,l,sigma2] = A[i][k,sigma1,l,sigma2]

                for k in range(chi_B):
                    for l in range(chi_B):
                        S_i[chi_A + k,sigma1,chi_A + l,sigma2] = B[i][k,sigma1,l,sigma2]      
        S.append(S_i)

    #Ajout du dernier tenseur
    S_i = np.zeros((chi_S,d,d), dtype="complex_")
    for sigma1 in range(d):
        for sigma2 in range(d):

            for k in range(chi_A):
                S_i[k,sigma1,sigma2] = A[n-1][k,sigma1,sigma2]

            for k in range(chi_B):
                S_i[k+chi_A,sigma1,sigma2] = B[n-1][k,sigma1,sigma2]
    S.append(S_i)
    
    
    return S


In [119]:
def pauli_to_MPO(H): #H est un hamiltonien donné sous forme d'une somme de produits de tenseurs de Pauli. Ici H est représenté sous forme de tableau où les termes en d'indice i dans la somme sont à la ième ligne
    M=H[0]
    for i in range(len(H)-1):
        M=MPO_sum(M,H[i+1])
    return M

In [120]:
H1=[[0.25*H2g,0.25*H2,0.25*H2,0.25*H2d],
            [0.25*Hzg,0.25*Hz,0.25*H2,0.25*H2d],
            [0.25*Hzg,0.25*H2,0.25*H2,0.25*H2d],
            [0.25*H2g,0.25*Hz,0.25*H2,0.25*H2d],
            [0.5*Hxg,0.5*Hz,0.5*Hx,0.5*H2d],
            [0.5*Hyg,0.5*Hz,0.5*Hy,0.5*H2d],
            [0.5*H2g,0.5*Hx,0.5*Hz,0.5*Hxd],
            [0.5*H2g,0.5*Hy,0.5*Hz,0.5*Hyd],
            [-0.5*H2g,-0.5*H2,-0.5*Hz,-0.5*H2d],
            [-0.5*H2g,-0.5*H2,-0.5*H2,-0.5*Hzd]]

In [121]:
H=pauli_to_MPO(H1)

In [122]:
H

[array([[[ 0.25+0.j ,  0.  +0.j ],
         [ 0.25+0.j ,  0.  +0.j ],
         [ 0.25+0.j ,  0.  +0.j ],
         [ 0.25+0.j ,  0.  +0.j ],
         [ 0.  +0.j ,  0.5 +0.j ],
         [ 0.  +0.j ,  0.  -0.5j],
         [ 0.5 +0.j ,  0.  +0.j ],
         [ 0.5 +0.j ,  0.  +0.j ],
         [-0.5 +0.j , -0.  +0.j ],
         [-0.5 +0.j , -0.  +0.j ]],
 
        [[ 0.  +0.j ,  0.25+0.j ],
         [ 0.  +0.j , -0.25+0.j ],
         [ 0.  +0.j , -0.25+0.j ],
         [ 0.  +0.j ,  0.25+0.j ],
         [ 0.5 +0.j ,  0.  +0.j ],
         [ 0.  +0.5j,  0.  +0.j ],
         [ 0.  +0.j ,  0.5 +0.j ],
         [ 0.  +0.j ,  0.5 +0.j ],
         [-0.  +0.j , -0.5 +0.j ],
         [-0.  +0.j , -0.5 +0.j ]]]),
 array([[[[ 0.25+0.j ,  0.  +0.j ],
          [ 0.  +0.j ,  0.  +0.j ],
          [ 0.  +0.j ,  0.  +0.j ],
          [ 0.  +0.j ,  0.  +0.j ],
          [ 0.  +0.j ,  0.  +0.j ],
          [ 0.  +0.j ,  0.  +0.j ],
          [ 0.  +0.j ,  0.  +0.j ],
          [ 0.  +0.j ,  0.  +0.j ],
      

In [123]:
np.shape(M[1])

(10, 2, 10, 2)

In [124]:
contraction(M)

ValueError: axes don't match array

In [138]:

def MPS_orth_left(A): #Normalise le MPS à gauche en effectuant des décompositions QR successives
    M = A.copy()

    q,r = sc.linalg.qr(M[0])
    M[0] = q.copy()
    M[0].resize((d,chi))
    r_aux = r.copy()
    r_aux.resize((chi,chi))
    M[1] = np.tensordot(r_aux,M[1], axes = ([1],[0]))
   
    
    for i in range(1,N-1): #On ne s'occupe pas du dernier tenseur
        M[i] = np.reshape(M[i],(chi*d,chi))
        q,r = sc.linalg.qr(M[i], mode = "economic")
        M[i] = np.reshape(q,(chi,d,chi))
        M[i+1] = np.tensordot(r,M[i+1], axes = ([1],[0]))
        

    return M

def MPS_orth_right(A):

    M = A.copy()
    
    r,q = sc.linalg.qr(M[N-1])
    M[N-1] = q.copy()
    #M[N-1].resize((chi,d))
    #r_aux = r.copy()
    #r_aux.resize((chi,chi))
    #M[N-2] = np.tensordot(M[N-2],r_aux,1)
    M[N-2] = np.tensordot(M[N-2],r,1)
    

    for i in range(N-2,0,-1): #On ne s'occupe pas du premier tenseur
        
        M[i] = np.reshape(M[i],(chi,d*chi))
        r,q = sc.linalg.rq(M[i], mode = "economic")
        M[i] = np.reshape(q,(chi,d,chi))
        M[i-1] = np.tensordot(M[i-1],r,1)

    
    return M

print(MPS_orth_right(A))
        

[array([[ 1.07961605e+00+6.01029822e-01j, -1.56788660e-16+4.22651913e-16j,
         1.50313217e-18-1.49366195e-16j,  9.24073919e-17+1.01477743e-16j,
         4.69887204e-33+1.06618653e-31j],
       [ 0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j]]), array([[[ 8.39745086e-01+2.69876374e-01j,
         -6.38214122e-16-1.42146675e-16j,
         -6.14585136e-18-8.18466984e-17j,
          6.83401071e-17+7.29467478e-17j,
          5.75274178e-32+9.26873307e-32j],
        [-1.20586426e-15+6.58997119e-16j,
         -4.44144618e-01+1.57259316e-01j,
         -9.66146266e-19+9.35115773e-17j,
         -6.58472954e-17-9.68783631e-17j,
         -3.79945796e-31-8.30633999e-32j]],

       [[ 5.16856929e-02-4.15948030e-01j,
          1.42127971e-05+1.14094740e-05j,
         -1.58330041e-01+2.96938279e-01j,
         -2.54211356e-01-2.75339142e-02j,
         -8.97993331e-17-

In [139]:

A = MPS_orth_right(A)


def right_contraction(Hr, Mt, Mb, H):
    
    Taux = np.tensordot(Hr,Mb, axes = ([2],[2]))
    Taux = np.tensordot(Taux,H, axes = ([3,0],[3,2]))
    Taux = np.tensordot(Taux,Mt, axes = ([0,3],[0,1]))
    Taux = np.transpose(Taux,(1,2,0))  #ordre final: de bas en haut: 2,0,1 (sens horaire en partant du milieu)

    return Taux

def left_contraction(Hl,Mt,Mb,H):

    Taux = np.tensordot(Hl,Mb, axes = ([2],[0]))
    Taux = np.tensordot(Taux,H, axes = ([1,2],[0,3]))
    Taux = np.tensordot(Taux,Mt, axes = ([0,2],[2,1]))
    Taux = np.transpose(Taux,(2,1,0)) #ordre final: de haut en bas: 0, 1, 2

    return Taux

#Calcul de R[i], composante à droite de la matrice effective (Rq: on ne calcule pas R[0] ici)

R = [0]*N #liste contenant les R_i
L = [0]*N


Taux = np.tensordot(H[N-1],A[N-1], axes = ([2],[1]))
Taux = np.tensordot(Taux,A[N-1].conj().T, axes = ([1],[0]))
R[N-2] = np.transpose(Taux,(0,2,1))
print(R[N-2].shape)

Taux = np.tensordot(H[0],A[0], axes = ([2],[0]))
Taux = np.tensordot(Taux,A[0].conj().T, axes = ([0],[1]))
L[1] = np.transpose(Taux,(2,0,1))
print(L[1].shape)


for i in range(N-3,-1,-1):
    R[i] = right_contraction(R[i+1],A[i+1].conj().T,A[i+1],H[i+1])

for i in range(2,N):
    L[i] = left_contraction(L[i-1],A[i-1].conj().T,A[i-1],H[i-1])





(10, 5, 5)
(5, 10, 5)


In [140]:
print(L[1].shape)
print(R[1].shape)

def effective_Matrix(L,R,H): #Calcule la matrice à diagonaliser
        Meff = np.tensordot(L,H,axes = ([1],[0]))
        Meff = np.tensordot(Meff,R, axes = ([3],[0]))
        Meff = np.transpose(Meff, (0,2,4,1,3,5))
        Meff = np.reshape(Meff,(chi*chi*d,chi*chi*d))
        return Meff

print(effective_Matrix(L[1],R[1],H[1]))
    

(5, 10, 5)
(10, 5, 5)
[[-1.16964582e-01+7.79241054e-019j  7.76211247e-17+1.32880574e-017j
  -3.73992894e-34-2.16027842e-033j ... -2.98457263e-49-1.13796751e-049j
   8.33876148e-50+7.70578629e-049j  5.90446892e-64+5.86820695e-064j]
 [ 7.76211247e-17-1.32880574e-017j  7.14714369e-03+3.15436678e-019j
   6.33567996e-19-2.33424569e-018j ... -1.50270631e-64-2.24187821e-064j
  -5.39334347e-64+5.33704779e-064j -2.53237933e-79-2.73234067e-078j]
 [-3.73992894e-34+2.16027842e-033j  6.33567996e-19+2.33424569e-018j
   2.36218522e-02-1.93091639e-019j ...  2.65178274e-33+2.09118410e-034j
  -2.47406484e-33-5.96167250e-033j  1.01555191e-47+1.24219416e-047j]
 ...
 [-2.98457263e-49+1.13796751e-049j -7.43060892e-65+3.13470932e-064j
   2.65178274e-33-2.09118410e-034j ...  6.19221386e-65-4.72195455e-097j
  -6.83255016e-65-1.33823705e-064j  2.58410406e-79+2.69688313e-079j]
 [ 8.33876148e-50-7.70578629e-049j -5.39334347e-64-5.08054358e-064j
  -2.47406484e-33+5.96167250e-033j ... -6.83255016e-65+1.33823705e-06

In [141]:
def DMRG(Np):
    E = [] #liste des énergies

    for _ in range(Np):

        #Balayage de gauche à droite
       
        #Traitement du 1er tenseur (cas de bord: les dimensions de A[0] sont différentes)

        Meff = np.tensordot(H[0],R[0],axes = ([1],[0]))
        Meff = np.transpose(Meff, (0,2,1,3))
        Meff = np.reshape(Meff, (chi*d,chi*d))

        # Diagonalisation (Lanczos)
        val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[0]) 
        vec = np.reshape(vec,(d,chi))
        E.append(val[0])

        #Décomposition SVD sur la matrice obtenue
        U,s,V = sc.linalg.svd(vec,full_matrices = False)
        S = np.diag(s)
        V = np.matmul(S,V)

        # Mise à jour des tenseurs A[0] et A[1]
        
        A[0] = U.copy()
        A[0].resize((d,chi))
        Vaux = V.copy()
        Vaux.resize((chi,chi))
        A[1] = np.tensordot(Vaux,A[1],1)

        #Calcul de L[1] 

        Taux = np.tensordot(H[0],A[0], axes = ([2],[0]))
        Taux = np.tensordot(Taux,A[0].conj().T, axes = ([0],[1]))
        L[1] = np.transpose(Taux,(1,0,2))
        

        
        
        for i in range(1,N-1): #balayage de gauche à droite (pour les tenseurs centraux)
            Meff = effective_Matrix(L[i],R[i],H[i])

            # Diagonalisation (Lanczos)
            val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[i]) 
            vec = np.reshape(vec,(chi*d,chi))
            E.append(val[0])

            #Décomposition SVD sur la matrice obtenue
            U,s,V = sc.linalg.svd(vec,full_matrices = False)
            S = np.diag(s)
            V = np.matmul(S,V)
        
            # Mise à jour des tenseurs A[i] et A[i+1]
        
            A[i] = np.reshape(U,(chi,d,chi))
            A[i+1] = np.tensordot(V,A[i+1],1)

            #Calcul de L[i] 
            L[i+1] = left_contraction(L[i],A[i].conj().T,A[i],H[i])

        #Traitement du dernier tenseur


        #Balayage de droite à gauche
        
        #Traitement du dernier tenseur
        
        Meff = np.tensordot(L[N-1],H[N-1],axes = ([1],[0]))
        Meff = np.transpose(Meff, (0,2,1,3))
        Meff = np.reshape(Meff, (chi*d,chi*d))

        # Diagonalisation (Lanczos)
        val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[0]) 
        vec = np.reshape(vec,(d,chi))
        E.append(val[0])

        # Décomposition SVD sur la matrice obtenue
        U,s,V = sc.linalg.svd(vec,full_matrices = False)
        S = np.diag(s)
        V = np.matmul(S,V)

        # Mise à jour des tenseurs A[N-1] et A[N-2]
        
        A[N-1] = V.copy()
        A[N-1].resize((chi,d))
        Uaux = U.copy()
        Uaux.resize((chi,chi))
        A[N-2] = np.tensordot(A[N-2],Uaux,1)
    

        #Calcul de R[N-2] 
        Taux = np.tensordot(H[N-1],A[N-1], axes = ([2],[1]))
        Taux = np.tensordot(Taux,A[N-1].conj().T, axes = ([1],[0]))
        R[N-2] = np.transpose(Taux,(0,2,1))

        

        for i in range(N-2,0,-1): #balayage de droite à gauche (pour les MPS centraux)
    
            Meff = effective_Matrix(L[i],R[i],H[i])

            # Diagonalisation (Lanczos)
            val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[i]) 
            vec = np.reshape(vec,(chi,d*chi))
            E.append(val[0])

            #Décomposition SVD sur la matrice obtenue
            U,s,V = sc.linalg.svd(vec,full_matrices = False)
            S = np.diag(s)
            U = np.matmul(U,S)
        
            # Mise à jour des tenseurs A[i] et A[i+1]
        
            A[i] = np.reshape(V,(chi,d,chi))
            A[i-1] = np.tensordot(A[i-1],U,1)

            #Calcul de R[i-1] 
            R[i-1] = right_contraction(R[i],A[i].conj().T,A[i],H[i])
        
    return E

print(DMRG(10))

        
        

[-0.1198665890963074, -0.21438764667508778, -0.21438764667508728, -0.3822538936964527, -0.21438764667508747, -0.2143876466750871, -0.144212869983291, -0.21438764667508775, -0.21438764667508758, -0.3822538936964528, -0.21438764667508747, -0.21438764667508733, -0.1426909759730055, -0.2143876466750876, -0.21438764667508747, -0.382253893696452, -0.21438764667508772, -0.21438764667508725, -0.11799247403741782, -0.21438764667508756, -0.21438764667508756, -0.38225389369645274, -0.2143876466750878, -0.2143876466750877, -0.15462437416918742, -0.2143876466750877, -0.21438764667508756, -0.38225389369645324, -0.2143876466750878, -0.214387646675088, -0.12906815055507306, -0.2143876466750877, -0.21438764667508756, -0.38225389369645335, -0.21438764667508736, -0.21438764667508686, -0.11585382415177144, -0.2143876466750872, -0.21438764667508747, -0.3822538936964533, -0.21438764667508753, -0.2143876466750875, -0.13459422049680048, -0.21438764667508764, -0.21438764667508764, -0.38225389369645235, -0.2143